## Creates the **Epidemiological Timeline** for a specific region by using the available data on WNV Cases ##

In [1]:
import pandas as pd
import numpy as np

enc = 'utf-8'
dec = 'greek8'

pd.options.display.max_columns = None
pd.options.display.max_rows = 10

In [2]:
NUTS0 = 'GR'
NUTS2 = 'Thessaly'
NUTS2_EL = 'θεσσαλιας'

In [3]:
data = pd.read_csv(f'../../data/{NUTS0}_WNV_cases_2010-2023_processed.csv', encoding = enc)
data.head(5)

,onset of symptoms,year,month,day,nuts2,nuts3,cases
0,2010-07-06,2010,7,6,κεντρικης μακεδονιας,σερρων,1
1,2010-07-16,2010,7,16,κεντρικης μακεδονιας,κιλκις,1
2,2010-07-18,2010,7,18,κεντρικης μακεδονιας,πελλας,1
3,2010-07-19,2010,7,19,κεντρικης μακεδονιας,θεσσαλονικης,2
4,2010-07-20,2010,7,20,κεντρικης μακεδονιας,ημαθιας,1


In [4]:
data.shape

(1431, 7)

In [5]:
df = data[data['nuts2'].isin([NUTS2_EL])].copy()

In [6]:
df.reset_index(drop = True, inplace = True)

In [7]:
df.drop(columns = ['onset of symptoms', 'day'], inplace = True)

In [8]:
df

,year,month,nuts2,nuts3,cases
0,2010,7,θεσσαλιας,λαρισας,1
1,2010,8,θεσσαλιας,λαρισας,1
2,2010,8,θεσσαλιας,λαρισας,1
3,2010,8,θεσσαλιας,λαρισας,1
4,2010,8,θεσσαλιας,λαρισας,1
...,...,...,...,...,...
158,2023,9,θεσσαλιας,τρικαλων,1
159,2023,9,θεσσαλιας,τρικαλων,3
160,2023,9,θεσσαλιας,τρικαλων,1
161,2023,10,θεσσαλιας,τρικαλων,1


In [9]:
df.cases.sum()

210

In [10]:
df_grouped = df.groupby(['year', 'month', 'nuts2', 'nuts3'], as_index = False)
df = df_grouped.sum()

In [11]:
df

,year,month,nuts2,nuts3,cases
0,2010,7,θεσσαλιας,λαρισας,1
1,2010,8,θεσσαλιας,λαρισας,5
2,2010,9,θεσσαλιας,λαρισας,3
3,2011,7,θεσσαλιας,καρδιτσας,3
4,2011,7,θεσσαλιας,λαρισας,4
...,...,...,...,...,...
38,2023,9,θεσσαλιας,λαρισας,7
39,2023,9,θεσσαλιας,τρικαλων,10
40,2023,10,θεσσαλιας,καρδιτσας,1
41,2023,10,θεσσαλιας,λαρισας,3


In [12]:
df.cases.sum()

210

In [13]:
print(np.sort(df.cases.unique()))

[ 1  2  3  4  5  6  7 10 14 15 19 23]


In [14]:
wnv_lau1_list = df['nuts3'].drop_duplicates().sort_values().tolist()

with open(f'../../data/{NUTS2}/{NUTS0}_{NUTS2}_WNV_2010-2023_NUTS3_Units_Processed.txt', 'w', encoding=enc) as f:
    for item in wnv_lau1_list:
        f.write("%s\n" % item)
  
print(f"Number of NUTS3 Units in {NUTS2} (from cases): {len(wnv_lau1_list)}")

Number of NUTS3 Units in Thessaly (from cases): 4


In [15]:
df['date_string'] = df['month'].astype(str) + '-' + df['year'].astype(str)

# Convert 'date_string' column to datetime format
df['dt_placement'] = pd.to_datetime(df['date_string'], format='%m-%Y').dt.to_period('M')
df.drop(columns=['date_string'], inplace = True)

In [16]:
df

,year,month,nuts2,nuts3,cases,dt_placement
0,2010,7,θεσσαλιας,λαρισας,1,2010-07
1,2010,8,θεσσαλιας,λαρισας,5,2010-08
2,2010,9,θεσσαλιας,λαρισας,3,2010-09
3,2011,7,θεσσαλιας,καρδιτσας,3,2011-07
4,2011,7,θεσσαλιας,λαρισας,4,2011-07
...,...,...,...,...,...,...
38,2023,9,θεσσαλιας,λαρισας,7,2023-09
39,2023,9,θεσσαλιας,τρικαλων,10,2023-09
40,2023,10,θεσσαλιας,καρδιτσας,1,2023-10
41,2023,10,θεσσαλιας,λαρισας,3,2023-10


In [17]:
df.cases.sum()

210

In [18]:
case_months = df['month'].drop_duplicates().sort_values().tolist()
case_months

[7, 8, 9, 10]

In [19]:
case_years = df['year'].drop_duplicates().sort_values().tolist()
case_years

[2010, 2011, 2018, 2019, 2020, 2022, 2023]

In [20]:
with open(f'../../data/{NUTS2}/{NUTS0}_{NUTS2}_WNV_2010-2023_NUTS3_Cases_Dates.txt', 'w', encoding=enc) as f:

    f.write("%s: " % 'months')
    for item in case_months:
        f.write("%s, " % item)

    f.write("\n")

    f.write("%s: " % 'years')
    for item in case_years:
        f.write("%s, " % item)

In [21]:
df.drop(columns=['month', 'year'], inplace = True)

In [22]:
## Creating DataFrame to hold negative examples

df_timeline = pd.DataFrame(columns = ['nuts2', 'nuts3', 'dt_placement', 'cases'])
timeline = []

for nuts3 in df['nuts3'].drop_duplicates().sort_values():
    for year in case_years:
        for month in case_months:
            timeline.append({'nuts2' : NUTS2_EL, 'nuts3' : nuts3, 'dt_placement' : f"{year:04}-{month:02}", 'cases' : 0})
            
df_timeline = pd.DataFrame(timeline)

In [23]:
df_timeline

,nuts2,nuts3,dt_placement,cases
0,θεσσαλιας,καρδιτσας,2010-07,0
1,θεσσαλιας,καρδιτσας,2010-08,0
2,θεσσαλιας,καρδιτσας,2010-09,0
3,θεσσαλιας,καρδιτσας,2010-10,0
4,θεσσαλιας,καρδιτσας,2011-07,0
...,...,...,...,...
107,θεσσαλιας,τρικαλων,2022-10,0
108,θεσσαλιας,τρικαλων,2023-07,0
109,θεσσαλιας,τρικαλων,2023-08,0
110,θεσσαλιας,τρικαλων,2023-09,0


In [24]:
df_timeline['dt_placement']= pd.to_datetime(df_timeline['dt_placement'], format='%Y-%m').dt.to_period('M')

In [25]:
df.reset_index(inplace = True, drop=True)
df_timeline.reset_index(inplace = True, drop=True)

In [26]:
rearranged_cols = ['nuts2',	'nuts3', 'dt_placement', 'cases']

# Reindex the DataFrame with the desired column order
df = df.reindex(columns=rearranged_cols)

In [27]:
df

,nuts2,nuts3,dt_placement,cases
0,θεσσαλιας,λαρισας,2010-07,1
1,θεσσαλιας,λαρισας,2010-08,5
2,θεσσαλιας,λαρισας,2010-09,3
3,θεσσαλιας,καρδιτσας,2011-07,3
4,θεσσαλιας,λαρισας,2011-07,4
...,...,...,...,...
38,θεσσαλιας,λαρισας,2023-09,7
39,θεσσαλιας,τρικαλων,2023-09,10
40,θεσσαλιας,καρδιτσας,2023-10,1
41,θεσσαλιας,λαρισας,2023-10,3


In [28]:
df_timeline

,nuts2,nuts3,dt_placement,cases
0,θεσσαλιας,καρδιτσας,2010-07,0
1,θεσσαλιας,καρδιτσας,2010-08,0
2,θεσσαλιας,καρδιτσας,2010-09,0
3,θεσσαλιας,καρδιτσας,2010-10,0
4,θεσσαλιας,καρδιτσας,2011-07,0
...,...,...,...,...
107,θεσσαλιας,τρικαλων,2022-10,0
108,θεσσαλιας,τρικαλων,2023-07,0
109,θεσσαλιας,τρικαλων,2023-08,0
110,θεσσαλιας,τρικαλων,2023-09,0


In [29]:
merged_timeline = pd.merge(df_timeline, df, how ='left', on=['nuts2','nuts3','dt_placement'])

In [30]:
merged_timeline

,nuts2,nuts3,dt_placement,cases_x,cases_y
0,θεσσαλιας,καρδιτσας,2010-07,0,NaN
1,θεσσαλιας,καρδιτσας,2010-08,0,NaN
2,θεσσαλιας,καρδιτσας,2010-09,0,NaN
3,θεσσαλιας,καρδιτσας,2010-10,0,NaN
4,θεσσαλιας,καρδιτσας,2011-07,0,3.0
...,...,...,...,...,...
107,θεσσαλιας,τρικαλων,2022-10,0,NaN
108,θεσσαλιας,τρικαλων,2023-07,0,NaN
109,θεσσαλιας,τρικαλων,2023-08,0,7.0
110,θεσσαλιας,τρικαλων,2023-09,0,10.0


In [31]:
merged_timeline['cases_y'] = merged_timeline['cases_y'].fillna(0)

In [32]:
merged_timeline['cases'] = merged_timeline['cases_x'] + merged_timeline['cases_y']
merged_timeline = merged_timeline.drop(columns=['cases_x', 'cases_y'])
merged_timeline['cases'] = merged_timeline['cases'].astype(int)

In [33]:
merged_timeline

,nuts2,nuts3,dt_placement,cases
0,θεσσαλιας,καρδιτσας,2010-07,0
1,θεσσαλιας,καρδιτσας,2010-08,0
2,θεσσαλιας,καρδιτσας,2010-09,0
3,θεσσαλιας,καρδιτσας,2010-10,0
4,θεσσαλιας,καρδιτσας,2011-07,3
...,...,...,...,...
107,θεσσαλιας,τρικαλων,2022-10,0
108,θεσσαλιας,τρικαλων,2023-07,0
109,θεσσαλιας,τρικαλων,2023-08,7
110,θεσσαλιας,τρικαλων,2023-09,10


In [34]:
merged_timeline['cases'].value_counts().sort_index()

0     69
1     11
2      8
3      4
4      5
      ..
10     1
14     1
15     2
19     1
23     1
Name: cases, Length: 13, dtype: int64

In [35]:
merged_timeline

,nuts2,nuts3,dt_placement,cases
0,θεσσαλιας,καρδιτσας,2010-07,0
1,θεσσαλιας,καρδιτσας,2010-08,0
2,θεσσαλιας,καρδιτσας,2010-09,0
3,θεσσαλιας,καρδιτσας,2010-10,0
4,θεσσαλιας,καρδιτσας,2011-07,3
...,...,...,...,...
107,θεσσαλιας,τρικαλων,2022-10,0
108,θεσσαλιας,τρικαλων,2023-07,0
109,θεσσαλιας,τρικαλων,2023-08,7
110,θεσσαλιας,τρικαλων,2023-09,10


In [36]:
merged_timeline.cases.sum()

210

In [37]:
merged_timeline.nuts2.unique()

array(['θεσσαλιας'], dtype=object)

In [38]:
merged_timeline.nuts3.unique()

array(['καρδιτσας', 'λαρισας', 'μαγνησιας', 'τρικαλων'], dtype=object)

In [39]:
merged_timeline.drop(columns=['nuts2'], inplace = True)
merged_timeline.rename(columns={'nuts3': 'NUTS3_NAME'}, inplace= True)

In [40]:
merged_timeline['NUTS3_NAME'] = merged_timeline['NUTS3_NAME'].apply(lambda x : x.replace('καρδιτσας','καρδιτσα, τρικαλα'))
merged_timeline['NUTS3_NAME'] = merged_timeline['NUTS3_NAME'].apply(lambda x : x.replace('λαρισας','λαρισα'))
merged_timeline['NUTS3_NAME'] = merged_timeline['NUTS3_NAME'].apply(lambda x : x.replace('μαγνησιας','μαγνησια, σποραδες'))
merged_timeline['NUTS3_NAME'] = merged_timeline['NUTS3_NAME'].apply(lambda x : x.replace('τρικαλων','καρδιτσα, τρικαλα'))

In [41]:
merged_timeline['NUTS2_NAME'] = 'θεσσαλια'

In [42]:
merged_timeline = merged_timeline[['NUTS2_NAME', 'NUTS3_NAME', 'dt_placement', 'cases']]
merged_timeline

,NUTS2_NAME,NUTS3_NAME,dt_placement,cases
0,θεσσαλια,"καρδιτσα, τρικαλα",2010-07,0
1,θεσσαλια,"καρδιτσα, τρικαλα",2010-08,0
2,θεσσαλια,"καρδιτσα, τρικαλα",2010-09,0
3,θεσσαλια,"καρδιτσα, τρικαλα",2010-10,0
4,θεσσαλια,"καρδιτσα, τρικαλα",2011-07,3
...,...,...,...,...
107,θεσσαλια,"καρδιτσα, τρικαλα",2022-10,0
108,θεσσαλια,"καρδιτσα, τρικαλα",2023-07,0
109,θεσσαλια,"καρδιτσα, τρικαλα",2023-08,7
110,θεσσαλια,"καρδιτσα, τρικαλα",2023-09,10


In [43]:
merged_timeline.cases.sum()

210

In [44]:
grouped = merged_timeline.groupby(['NUTS2_NAME', 'NUTS3_NAME',	'dt_placement'], as_index = False)
timeline_grouped = grouped.sum()
timeline_grouped

,NUTS2_NAME,NUTS3_NAME,dt_placement,cases
0,θεσσαλια,"καρδιτσα, τρικαλα",2010-07,0
1,θεσσαλια,"καρδιτσα, τρικαλα",2010-08,0
2,θεσσαλια,"καρδιτσα, τρικαλα",2010-09,0
3,θεσσαλια,"καρδιτσα, τρικαλα",2010-10,0
4,θεσσαλια,"καρδιτσα, τρικαλα",2011-07,3
...,...,...,...,...
79,θεσσαλια,"μαγνησια, σποραδες",2022-10,0
80,θεσσαλια,"μαγνησια, σποραδες",2023-07,0
81,θεσσαλια,"μαγνησια, σποραδες",2023-08,0
82,θεσσαλια,"μαγνησια, σποραδες",2023-09,0


In [45]:
timeline_grouped[['NUTS2_NAME', 'NUTS3_NAME', 'dt_placement']].value_counts().sort_values(ascending=False)

NUTS2_NAME  NUTS3_NAME          dt_placement
θεσσαλια    καρδιτσα, τρικαλα   2010-07         1
                                2018-09         1
                                2010-09         1
                                2010-10         1
                                2011-07         1
                                               ..
            μαγνησια, σποραδες  2023-07         1
                                2023-08         1
                                2023-09         1
                                2018-08         1
                                2023-10         1
Length: 84, dtype: int64

In [46]:
timeline_grouped.cases.sum()

210

In [47]:
timeline_grouped.to_csv(f"../../data/{NUTS2}/{NUTS0}_{NUTS2}_WNV_Cases_NUTS3_2010-2023.csv", encoding = enc, index = False)